# Residual Volatility (RESVOL) — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**RESVOL** style factor exactly as MSCI describes it in *Barra USE4 Empirical Notes*
(Appendix A, p. 52; §3.4, p. 16). It is **not** a research notebook. The deliverable
is a clean, USE4-faithful RESVOL factor written to parquet, suitable for input to a
multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- Daily price/return data (Sharadar SEP cleaned parquet)
- Ken French daily RF (for excess return computation)
- ESTU monthly universe (from `estu_monthly.parquet`)
- Beta factor output (`beta_use4.parquet`) — for Stage 6 orthogonalization only

**Deliverable:** A parquet file `resvol_use4.parquet` keyed by
`(permaticker, signal_date)` containing the standardized RESVOL exposure for
every stock in the coverage universe on every monthly signal date.

**Companion specs:**
- `02_style_factors/beta/beta_spec.ipynb` — Beta (upstream; provides `beta_score` for orthogonalization)
- `02_style_factors/nlb/nlb_spec.ipynb` — NLB (also orthogonalizes to Beta; same pattern)

## §1. The USE4 RESVOL spec — verbatim quotes

### 1a. Barra USE4 Empirical Notes, Appendix A (p. 52) — formal descriptor definition

> **Residual Volatility (RESVOL)**
>
> *Definition:* `0.75 · DASTD + 0.15 · CMRA + 0.10 · HSIGMA`
>
> *DASTD (Daily standard deviation):* Exponentially weighted volatility of daily excess returns
> over the trailing T = 252 trading days, with a half-life of 42 trading days.
>
> *CMRA (Cumulative range):* Differentiates stocks that have experienced wide swings
> from those that have traded in a narrow range. Let Z(τ) be the cumulative excess log return
> over the past τ months, each month defined as 21 compounded trading days:
>
> `Z(τ) = Σ_{τ=1}^{T} [ln(1 + r_τ) − ln(1 + r_{fτ})],  T = 1, ..., 12`
>
> where r_τ is the 21-day compounded stock return for month τ and r_{fτ} is the risk-free return.
> The cumulative range is:
>
> `CMRA = ln(1 + Z_max) − ln(1 + Z_min)`
>
> where Z_max = max{Z(τ)}, Z_min = min{Z(τ)}, T = 1,...,12. No exponential decay is applied to CMRA.
>
> *HSIGMA (Historical sigma):* Volatility of the residual returns e_t from the market model:
>
> `r_t − r_{ft} = α + β · R_t + e_t,   σ = std(e_t)`
>
> Estimated over the trailing T = 252 trading days with a half-life of 63 trading days.

### 1b. Empirical Notes, §3.4 (p. 16) — orthogonalization

> *"The Residual Volatility factor is orthogonalized with respect to Beta to reduce
> collinearity between these two risk factors."* (paraphrase; see also Menchero 2010)

### 1c. Methodology Notes, §2.3 (p. 9) — standardization rule

> *"Descriptors are standardized to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

### 1d. Methodology Notes, §2.2 (p. 8) — outlier treatment

> *"We trim these observations to three standard deviations from the mean."*

---

**That is all the PDFs say about RESVOL construction.** The following are NOT covered:
exact RF source, minimum observation thresholds, treatment of stocks with < 252 days history,
precise definition of CMRA month boundaries, and the technical details of the
orthogonalization regression.

## §2. End-to-end pipeline

```
┌────────────────────────────────────────────────────────────────────┐
│  UPSTREAM:  estu_build.ipynb   →  estu_monthly.parquet             │
│             daily_build.ipynb  →  daily_returns.parquet (+ maps)   │
│             beta_build.ipynb   →  beta_use4.parquet                │
├────────────────────────────────────────────────────────────────────┤
│  STAGE 0:  Setup, parameters                                       │
│  STAGE 1:  Load shared daily-returns artifact                      │
│  STAGE 2:  Load shared ESTU                                        │
│  STAGE 3:  _td_to_sig map (benchmark already in daily.mkt_ret)     │
│  STAGE 4:  RESVOL = 0.75·DASTD + 0.15·CMRA + 0.10·HSIGMA           │
│  STAGE 5:  Outlier trim (3σ)                                       │
│  STAGE 6: Orthogonalize to Beta (√mcap WLS)                      │
│  STAGE 7:  Standardize (CW mean = 0, EW std = 1)                   │
│  STAGE 8:  Save deliverable                                        │
│  STAGE 9:  Validate                                                │
└────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** for any signal_date `t`, only data dated ≤ `t` is used.

**One-sweep run:** `your end-to-end runner` executes the whole
pipeline in dependency order (sequential, one kernel at a time).

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `resvol`
**Standardized score column:** `resvol_score`

**File:** `data/out/resvol_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |
| `resvol` | Float64 | **Raw composite descriptor** — 0.75·DASTD + 0.15·CMRA + 0.10·HSIGMA (post-clip, pre-ortho) |
| `resvol_score` | Float64 | **Final RESVOL exposure** — clipped, orthogonalized to Beta, then standardized cross-sectionally |
| `n_obs` | UInt32 | Number of daily observations used in estimation (= trading days in window; 252 for full window) |

**Coverage rules:**
- Compute `resvol` and `resvol_score` for **every stock with sufficient data**
  (`n_obs ≥ MIN_OBS`), not just ESTU members.
- Standardization moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the *same* standardization parameters applied so the final
  factor is comparable across the coverage universe.
- Because CMRA requires exactly 12 complete months (252 trading days), stocks with
  < 252 days of history are excluded entirely. `MIN_OBS = 252` is the effective floor.

## §4. Upstream factor inputs

This factor depends on the following pre-built factor outputs, which must be present
in `data/out/` before this build is run:

| Variable | File | Purpose |
|---|---|---|
| `beta` | `beta_use4.parquet` | Provides `beta_score` (standardized Beta exposure) for Stage 6 cross-sectional orthogonalization. The HSIGMA regression uses the same half-life (63 td) as Beta but is re-run from raw daily data — the Beta parquet itself is only needed for the orthogonalization step. |

## §5. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — RESVOL parameters:**

> `RESVOL_W_DASTD = 0.75` — *"0.75 · DASTD"* (Appendix A, p. 52)
>
> `RESVOL_W_CMRA = 0.15` — *"0.15 · CMRA"* (Appendix A, p. 52)
>
> `RESVOL_W_HSIGMA = 0.10` — *"0.10 · HSIGMA"* (Appendix A, p. 52)
>
> `DASTD_WINDOW_TD = 252` — *"trailing T = 252 trading days"* (DASTD)
>
> `DASTD_HALF_LIFE_TD = 42` — *"half-life of 42 trading days"* (DASTD)
>
> `CMRA_MONTHS = 12` — *"T = 1,...,12"* months (CMRA)
>
> `CMRA_DAYS_PER_MONTH = 21` — *"each month defined as 21 compounded trading days"* (CMRA)
>
> `HSIGMA_WINDOW_TD = 252` — *"trailing T = 252 trading days"* (HSIGMA)
>
> `HSIGMA_HALF_LIFE_TD = 63` — *"half-life of 63 trading days"* (HSIGMA; confirmed equal to Beta HL)

**NOT IN PDF:** Minimum observations threshold, calendar start date.

**SUGGESTION:** `MIN_OBS = 252` (full window required for CMRA; partial windows excluded).
`CALENDAR_START = date(1999, 1, 1)` (matches all other USE4 factor builds).

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
RESVOL_W_DASTD        = 0.75   # USE4 spec: Appendix A weight for DASTD
RESVOL_W_CMRA         = 0.15   # USE4 spec: Appendix A weight for CMRA
RESVOL_W_HSIGMA       = 0.10   # USE4 spec: Appendix A weight for HSIGMA

DASTD_WINDOW_TD       = 252    # USE4 spec: trailing window for DASTD (trading days)
DASTD_HALF_LIFE_TD    = 42     # USE4 spec: exponential half-life for DASTD

CMRA_MONTHS           = 12     # USE4 spec: number of 21-td months for CMRA
CMRA_DAYS_PER_MONTH   = 21     # USE4 spec: trading days per month for CMRA

HSIGMA_WINDOW_TD      = 252    # USE4 spec: trailing window for HSIGMA (trading days)
HSIGMA_HALF_LIFE_TD   = 63     # USE4 spec: half-life for HSIGMA (= Beta half-life)

# SUGGESTION (NOT IN PDF) — full window required: CMRA is undefined for < 12 complete months
MIN_OBS               = 252

# SUGGESTION (NOT IN PDF) — calendar start (matches all USE4 builds)
CALENDAR_START        = date(1999, 1, 1)

# -- Factor type (read by /build-factor to select boilerplate template)
FACTOR_TYPE = "time_series"   # "time_series" or "fundamental"
```

## §6. STAGE 1 — Load the shared daily panel

By now you have built this plumbing inline at least once (Beta). This factor
consumes the **shared daily panel** instead — the refactor step specified in
`01.5_daily/daily_spec.ipynb`:

| Variable | File (in `data/out/`) | Contents |
|---|---|---|
| `daily` | `daily_returns.parquet` | `permaticker, date, ret, mcap_lag, mkt_ret` — excess log returns; `mkt_ret` is the ESTU cap-weighted excess benchmark |
| `prices` | `ticker_permaticker.parquet` | ticker → permaticker map (spot-check lookups) |

**Build order:** `estu_build.ipynb` → `daily_build.ipynb` → this factor.

🧪 **VALIDATE:** performed once in the daily-panel build — benchmark correlation
> 0.90, crisis-vol sanity, load-path equivalence, missing-benchmark dates < 30.

## §7. STAGE 2 — Load the shared ESTU (`estu_monthly.parquet`)

✅ **PDF SPEC:** USE4 ESTU = MSCI USA Investable Markets Index (USA IMI) — ~99% of
US float-adjusted market cap, liquidity- and stability-screened, ~2,500–3,000 names.

**SUGGESTION:** load the shared `data/out/estu_monthly.parquet` built by
`estu_build.ipynb` (spec: `01_estu/estu_spec.ipynb`): top ~3,000
domestic common stocks on NYSE/NASDAQ/NYSEMKT, ATVR liquidity screen
(add ≥ 20%, retain ≥ 10%), 3,000/3,500 hysteresis buffer. **Every factor must use
this same ESTU** — factor-specific universes would break the multi-factor
cross-sectional regression.

🧪 **VALIDATE:**
- 330 monthly signal dates (1999-01-29 →); ESTU size mean ≈ 2,500, range ≈ 1,800–3,050
- Mega-caps (AAPL, MSFT, GOOGL, JPM) present every month they were public

## §8. STAGE 3 — ESTU benchmark (already in the daily artifact)

✅ **PDF SPEC:** factor regressions use the cap-weighted estimation-universe return.

**SUGGESTION:** `daily.mkt_ret` already carries the ESTU cap-weighted **excess**
benchmark, built and validated once in `daily_build.ipynb`. This stage only derives
`_td_to_sig` (trading day → owning signal date, via `factor_lib.make_td_to_sig`)
for the validation quintile checks.

🧪 **VALIDATE:** missing `mkt_ret` dates after the first signal date < 30.

## §9. STAGE 4 — RESVOL estimator

✅ **PDF SPEC (Barra USE4 Empirical Notes, Appendix A, p. 52):**

> **DASTD — Daily standard deviation**
>
> Exponentially weighted volatility of daily excess returns over the trailing
> T = 252 trading days, with a half-life of 42 trading days.
>
> `DASTD = sqrt( Σ_{d=1}^{252} w_d · (exc_ret_d)^2 / Σ w_d )`
>
> where `w_d = exp(−ln(2) / 42 · d)`, d=1 (most recent trading day) to 252 (oldest).

> **CMRA — Cumulative range**
>
> Partition the trailing 252 trading days into 12 non-overlapping buckets of 21 td each.
> For bucket τ, compute the cumulative excess log return:
>
> `Z(τ) = Σ_{k ∈ bucket τ} [ln(1 + r_k) − ln(1 + r_{fk})],  τ = 1,...,12`
>
> `CMRA = ln(1 + Z_max) − ln(1 + Z_min)`
>
> where Z_max = max{Z(τ)}, Z_min = min{Z(τ)}. No exponential decay applied to CMRA.

> **HSIGMA — Historical sigma**
>
> Run WLS market model over the trailing 252 trading days with HL = 63 trading days:
>
> `exc_ret_t = α + β · R_t + e_t`
>
> `HSIGMA = sqrt( Σ_{d=1}^{252} w_d · e_d^2 / Σ w_d )`
>
> where `w_d = exp(−ln(2) / 63 · d)`, same convention as DASTD but with HL=63.
> `R_t` is the ESTU cap-weighted market excess return from Stage 3.

> **Composite:**
>
> `resvol = 0.75 · DASTD + 0.15 · CMRA + 0.10 · HSIGMA`

**NOT IN PDF — DASTD/HSIGMA denominator:** The spec formula has `Σ w_d · r_d^2 / Σ w_d`
but does not specify whether to use a bias-corrected denominator.

**SUGGESTION:** Use `Σ w_d` as denominator (no bias correction). Matches standard Barra practice
and avoids introducing a per-stock correction factor.

**NOT IN PDF — CMRA with Z_min < −1:** If a stock lost more than 100% in cumulative terms
for some bucket (practically impossible but numerically possible with log returns near −1),
`ln(1 + Z_min)` is undefined.

**SUGGESTION:** Clip Z_min to −0.999 before taking the log. This is a guard for a degenerate
case that should never occur in practice with valid price data.

**NOT IN PDF — HSIGMA regression source:** HSIGMA's WLS market model uses HL=63, which
matches the Beta factor's half-life exactly. The Beta parquet stores `beta` and `beta_score`
but not per-day residuals, so we re-run the regression fresh from daily data.

**SUGGESTION:** Re-run the WLS market model independently for HSIGMA using HL=63 and T=252.
The Beta parquet (`beta_use4.parquet`) is only needed for Stage 6 orthogonalization.

**NOT IN PDF — Minimum observations:** The spec says T=252 for all three sub-descriptors
but does not define what happens when fewer days are available.

**SUGGESTION:** `MIN_OBS = 252`. A stock must have all 252 trading days present to receive a
RESVOL score. CMRA is the binding constraint — partial 12-month windows are excluded entirely.

---
*The section above (PDF SPEC quote through final SUGGESTION) is the STAGE4_DESCRIPTION that /build-factor will inject verbatim into the Stage 4 stub docstring. Content below this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**

- **DASTD:** Filter to the 252 most-recent trading days ≤ signal_date. Compute weights
  `w_d = exp(−ln(2)/42 · d)` with d=1 for most recent. `DASTD = sqrt(Σ(w_d · exc_ret_d²) / Σ w_d)`.

- **CMRA:** Partition the 252-day window into 12 non-overlapping 21-td buckets (most-recent first).
  Sum daily excess log returns within each bucket to get Z(τ).
  `CMRA = ln(1 + max(Z)) − ln(1 + max(min(Z), −0.999))`.

- **HSIGMA:** Run WLS market model `exc_ret = α + β · mkt_ret + e` with weights
  `w_d = exp(−ln(2)/63 · d)` over the same 252-day window.
  `HSIGMA = sqrt(Σ(w_d · e_d²) / Σ w_d)`.

- **PIT:** Only use trading days strictly ≤ signal_date.

🧪 **VALIDATE:**
- `resvol` is always positive (it is a composite of volatility measures)
- Median cross-sectional `resvol` should be in a plausible range (~0.005–0.030) for daily vol
- HSIGMA ≤ DASTD for most stocks (market model explains some variance; residual should be smaller)
- CMRA ≥ 0 always by construction (`Z_max ≥ Z_min`)
- Ticker spot check at a recent signal_date: TSLA, GME, NVDA → top quintile; JNJ, PG, KO → bottom quintile

## §10. STAGE 5 — Outlier treatment

**NOT IN PDF for RESVOL specifically.** Methodology Notes §2.2 (p. 8) describes a general
outlier-treatment algorithm that applies to all descriptors:

> *"We trim these observations to three standard deviations from the mean."*

**SUGGESTION:** Apply 3σ trimming per signal_date, computed within ESTU using
cap-weighted mean ± 3 × equal-weighted std. Applied to all stocks in the coverage universe.

**Note on distribution shape:** `resvol` is a volatility composite — always positive and
right-skewed. The lower clip bound can be negative in the formula but in practice rarely
activates. Expect ~0.5–2% of ESTU rows clipped at the upper bound per date.

**Pipeline position:** STAGE 5 (clip) is executed **before** STAGE 6 (orthogonalize).
This ensures the Beta orthogonalization is not distorted by extreme raw composite values.

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows clipped at the upper bound per date (right-skewed distribution)
- Negligible or zero clipping at the lower bound
- After trimming, cross-sectional skewness of `resvol` should be reduced relative to pre-clip

## §11. STAGE 6 — Orthogonalize to Beta

✅ **PDF SPEC (Empirical Notes §3.4, p. 16):**

> *"The Residual Volatility factor is orthogonalized with respect to Beta to reduce
> collinearity between these two risk factors."* (paraphrase; see also Menchero 2010)

The orthogonalized descriptor is the residual from a **cross-sectional WLS regression** of
the clipped `resvol` on `beta_score`, run per signal_date, restricted to ESTU members,
with weights w_i = √mcap_i.

**Formula:**

```
resvol_orth_{i,t} = resvol_{i,t} − γ_t · beta_score_{i,t}
```

where γ_t is estimated by no-intercept WLS with weights w_i = √mcap_{i,t}, over ESTU members only.
`beta_score` is the standardized Beta exposure from `beta_use4.parquet`.

**Applied to ALL stocks:** Once γ_t is estimated from ESTU, apply the same γ_t to non-ESTU
stocks as well (same cross-sectional correction for the full coverage universe).

**NOT IN PDF — Use `beta_score` (standardized) or raw `beta` (~1.0 scale)?**

**SUGGESTION:** Use `beta_score`. In the factor-model context, factors are defined in standardized
exposure space. Using raw beta (~1.0 for an average stock) would mix volatility units with
regression coefficient units and produce an uninterpretable γ_t.

**NOT IN PDF — Intercept in the orthogonalization regression?**

**SUGGESTION:** No intercept. We are projecting out only the Beta component; Stage 7 handles
demeaning. Including an intercept would inadvertently re-demean `resvol` before standardization.

🧪 **VALIDATE:**
- Cap-weighted Pearson correlation between `resvol_orth` and `beta_score` within ESTU ≈ 0
  (target: |ρ| < 0.05)
- Std of `resvol_orth` is slightly less than std of `resvol` before orthogonalization
  (orthogonalization removes the Beta-correlated variance component)
- No NaN introduced for stocks that had valid `resvol` after Stage 5

## §12. STAGE 7 — Standardize (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p. 9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per signal_date `t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (mcap_i · resvol_orth_i) / Σ mcap_i   # cap-weighted mean
σ_t  = std_{i ∈ ESTU_t}(resvol_orth_i)                        # equal-weighted std
resvol_score_{i,t} = (resvol_orth_i − μ_t) / σ_t              # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero RESVOL exposure.

**Input to this stage:** `resvol_orth` (the orthogonalized intermediate from Stage 6).
The output column is named `resvol_score`.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · resvol_score_i) / Σ mcap_i ≈ 0` (cap-weighted mean = 0 by construction)
- `std_{i ∈ ESTU}(resvol_score_i) ≈ 1` (equal-weighted std = 1 by construction)

## §13. STAGE 8 — Save deliverable

Write the final panel to `data/out/resvol_use4.parquet`.
Compression: zstd. Statistics: True.

**Column `resvol`** in the saved file is the **post-clip** raw composite (output of Stage 5),
not the pre-clip value and not the orthogonalized intermediate. The orthogonalized value
(`resvol_orth`) is transient — it is not saved to disk.

**Column `resvol_score`** is the final standardized exposure (output of Stage 7) and is
the primary deliverable for downstream risk model use.

## §14. STAGE 9 — Validation

### Required checks (all must pass before signing off)

**1. Standardization moments on ESTU:**
```
cap-weighted mean of resvol_score ≈ 0   (|mean| < 1e-6)
equal-weighted std of resvol_score ≈ 1   (|std − 1| < 0.02)
```

**2. Raw descriptor sanity:**
```
resvol > 0 for all valid rows (volatility composite is strictly positive)
Median cross-sectional resvol in [0.005, 0.030] on most signal dates
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null resvol_score per completed-month date post-2005
(the final in-progress signal date is excluded: freshest prices can lag the
Sharadar refresh and recent listings lack the full 252-day window)
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.90
(RESVOL is moderately persistent; expect ρ in the 0.90–0.95 range, similar to Beta)
```

**5. Economic direction:**
```
Q5 (high RESVOL): growth/speculative stocks, recent IPOs, small-cap high-beta names
Q1 (low RESVOL): large-cap defensives (utilities, consumer staples), low-beta blue chips
Spot checks: TSLA, GME, NVDA → Q5 on most recent dates;  JNJ, PG, KO → Q1
```

**6. Disk vs memory equivalence:**
```
max abs diff of resvol values between in-memory panel and read-back parquet < 1e-10
```

**7. Beta orthogonalization (bonus check):**
```
|cap-weighted correlation(resvol_orth, beta_score) within ESTU| < 0.05
(orthogonalization should eliminate the Beta-RESVOL correlation)
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §15. Common pitfalls — read this before coding

**Pitfall 1: Mixing DASTD and HSIGMA half-lives**
DASTD uses HL=42; HSIGMA uses HL=63. These produce different exponential weight vectors.
Using the same weight array for both is a silent error that produces plausible-looking but
incorrect output. Define two separate weight arrays `_w_dastd` and `_w_hsigma` and never share them.

**Pitfall 2: CMRA requires exactly 252 trading days**
CMRA is undefined if fewer than 12 complete 21-td months are available. A stock with 250 days
of history cannot compute CMRA and should receive no RESVOL score. If `MIN_OBS < 252`, you will
silently compute DASTD+HSIGMA-only composites for thin-window stocks — a spec deviation.

**Pitfall 3: CMRA buckets are non-overlapping slices, not a rolling window**
Month 1 = most-recent 21 trading days. Month 2 = the 21 days immediately before that.
Each day appears in exactly one bucket. Do not compute a 21-day rolling cumulative return
over the full 252-day series — that is a different and incorrect calculation.

**Pitfall 4: HSIGMA is the weighted std of residuals, not plain std**
`HSIGMA = sqrt(Σ(w_d · e_d²) / Σ w_d)` — the same denominator convention as DASTD.
Using `numpy.std(residuals)` (unweighted) will produce a numerically close but incorrect value.

**Pitfall 5: Orthogonalize to beta_score, not raw beta**
`beta_use4.parquet` contains both `beta` (raw coefficient, ~1.0 for average stock) and
`beta_score` (standardized exposure). Stage 6 must regress `resvol` on `beta_score`.
Using raw beta produces a numerically valid regression but mixes incompatible scales.

**Pitfall 6: Apply orthogonalization γ_t to ALL stocks, not just ESTU**
Estimate γ_t from ESTU members only (WLS, √mcap weights). Then subtract `γ_t · beta_score`
from every stock in the coverage universe, including non-ESTU stocks. Failing to apply
to non-ESTU stocks leaves them in a different factor space than ESTU members.

## §16. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/resvol_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3 exactly: 7 columns, correct types
2. ✅ All 7 validation checks in §14 pass within tolerance
3. ✅ ESTU has ~2,500–3,000 names per date, stable over time
4. ✅ Cap-weighted mean of `resvol_score` is machine-zero for every date in ESTU
5. ✅ Equal-weighted std of `resvol_score` is 1.0 (to 2 decimal places) for every date in ESTU
6. ✅ `resvol` (clipped composite) is strictly positive for all valid rows
7. ✅ Coverage of `resvol_score` across the full universe ≥ 4,000 stocks per completed-month date post-2005 (final in-progress date excluded)
8. ✅ Month-over-month rank stability ρ > 0.90
9. ✅ Every NOT IN PDF judgment call is documented in the build (comment or markdown)

**Once RESVOL is built and passes validation, the next steps are:**
- No USE4 style factors directly depend on RESVOL as an upstream input.
- RESVOL is a direct input to the USE4 multi-factor risk model alongside Beta, Momentum,
  Earnings Yield, Non-Linear Beta, Non-Linear Size, and the remaining style factors.
- Consider building the Size factor next (standalone, no upstream dependencies).